# Omniframes — the guided tour

**Dataframe code in, governed semantic queries out.** This notebook walks the whole feature
surface of [omniframes](https://github.com/exploreomni/omniframes) — read, aggregate, `HAVING`,
mixed aggregation, UDFs, raw SQL, cross-frame joins, totals, writers — and prints
`explain()` after every step, so you can always see *which tier* did the work:

| tier | what it is | where it runs |
|---|---|---|
| **1 · semantic** | a governed Omni query: model measures, topic join paths, row-level security | Omni |
| **2 · sql** | warehouse SQL over an embedded *governed* sub-query | the warehouse |
| **3 · local** | Arrow compute over the largest remote prefix omniframes could push down | this process |

## Two backends, one notebook

The setup cell below picks its backend from the environment:

* **Offline (the default).** With no credentials set, it runs against `FakeOmniAPI` — an
  in-process, wire-faithful fake of the Omni query API, executing real DuckDB queries over the
  deterministic bench dataset checked into `tests/data/bench/`. No network, no key, real answers.
* **Live.** Set `OMNI_BASE_URL` and `OMNI_API_KEY` and every cell below runs against your org,
  unchanged. Point it at a model with `OMNI_MODEL` / `OMNI_TOPIC` (defaults: `bench_ecommerce` /
  `order_items`), and — for the raw-SQL cells only — name the warehouse schema holding the bench
  tables with `OMNI_BENCH_SCHEMA` (e.g. `OMNIFRAMES_BENCH.`, trailing dot included).

```bash
export OMNI_BASE_URL="https://acme.omni.co"
export OMNI_API_KEY="…"                 # never paste a key into a cell
export OMNI_BENCH_SCHEMA="OMNIFRAMES_BENCH."
```

That parity is the point: the offline fake serves exactly the model documented in
`docs/bench_omni_model.md`, so an expectation that holds here holds live.

In [ ]:
"""Build a session — against a live org if credentials are set, else against the fake."""

import os
import sys
from pathlib import Path

import omniframes as of
from omniframes import functions as F

MODEL = os.environ.get("OMNI_MODEL", "bench_ecommerce")
TOPIC = os.environ.get("OMNI_TOPIC", "order_items")
SCHEMA = os.environ.get("OMNI_BENCH_SCHEMA", "")  # raw-SQL cells only; "" offline
LIVE = bool(os.environ.get("OMNI_BASE_URL") and os.environ.get("OMNI_API_KEY"))

if LIVE:
    # The key is read from OMNI_API_KEY and lives inside the transport — it never appears in a
    # repr, a log line or an error message, and it must never appear in a notebook cell either.
    session = of.OmniSession.builder.base_url_from_env().api_key_from_env().get_or_create()
    backend = f"live org at {os.environ['OMNI_BASE_URL']}"
else:
    import httpx

    from omniframes.transport import HttpTransport

    # tests/ is a dev-only package (it ships with the repository, not with the wheel), so put
    # the checkout root on sys.path.  Run this notebook from inside the repository.
    here = Path.cwd().resolve()
    root = next(
        (
            candidate
            for candidate in (here, *here.parents)
            if (candidate / "pyproject.toml").exists() and (candidate / "tests" / "fakes").is_dir()
        ),
        None,
    )
    if root is None:
        raise RuntimeError(
            f"no omniframes checkout found at or above {here}. Either run this notebook from "
            "inside the repository, or set OMNI_BASE_URL and OMNI_API_KEY to use a live org."
        )
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

    from tests.fakes import DEFAULT_TOKEN, FakeOmniAPI

    BASE_URL = "https://bench.example.omni.co"
    handler = FakeOmniAPI(model_name=MODEL)
    client = httpx.Client(transport=httpx.MockTransport(handler), base_url=BASE_URL)
    session = (
        of.OmniSession.builder.base_url(BASE_URL)
        .transport(HttpTransport(base_url=BASE_URL, api_key=DEFAULT_TOKEN, client=client))
        .get_or_create()
    )
    backend = "in-process FakeOmniAPI (no network, no credentials)"

print(f"omniframes {of.__version__} → {backend}")
print(f"model={MODEL!r}  topic={TOPIC!r}")

# Building a session performs no I/O at all; the whoami preflight runs lazily, once, before the
# first call that needs the network.
orders = session.read.topic(MODEL, TOPIC)

REVENUE = "order_items.total_sale_price"  # governed measure: SUM(sale_price)
ORDERS = "order_items.count"  # governed measure: COUNT(*)

---

## 1 · Read a topic

A **topic** carries the model's join paths, so fields from any joined view (`users.*`,
`products.*`) are selectable without saying how they join. Everything below is lazy: nothing
touches the network until `.show()`.

In [ ]:
recent = (
    orders.select("order_items.id", "users.state", "order_items.status", "order_items.sale_price")
    .filter(F.col("order_items.status").isin("complete", "shipped"))
    .sort(F.col("order_items.id").desc())
    .limit(5)
)
recent.show()

In [ ]:
print(recent.explain())

One governed query, `(none — fully pushed down)`: the projection, the `IN` filter, the sort and
the limit all rode the wire. Note `limit: 5` — omniframes **always** sends an explicit limit
(`50000` by default), so nothing ever streams unbounded by accident.

---

## 2 · Aggregate with governed measures

`F.measure(...)` references a measure **defined in the Omni model**. Omni computes it, under its
own definition, with the model's joins and security applied — omniframes never emulates one
locally.

`group_by().agg()` is sugar: in Omni, selecting dimensions alongside measures *is* the group-by,
and `orders.select("users.state", F.measure(REVENUE))` compiles to the identical query.

`.grain("month")` buckets a timestamp server-side; `.alias(...)` renames client-side (the wire
has no aliasing) and can be used in the `sort` that follows.

In [ ]:
month = F.col("order_items.created_at").grain("month").alias("month")

monthly = (
    orders.group_by(month, F.col("users.state").alias("state"))
    .agg(F.measure(REVENUE).alias("revenue"), F.measure(ORDERS).alias("orders"))
    .filter(F.col("users.state").isin("California", "New York"))
    .sort(F.col("month").desc(), F.col("state"))
    .limit(6)
)
monthly.show()

In [ ]:
print(monthly.explain())

Still tier 1. The `aliases:` line is the whole aliasing story: `order_items.created_at[month]`
goes on the wire, `month` comes back to you.

---

## 3 · Filter *after* aggregating — a real `HAVING`

A predicate on a governed measure is not a client-side pass. It compiles to a measure-keyed wire
filter, which the server applies after the `GROUP BY` — a genuine `HAVING`. It works whether the
measure is selected or not, and whether you write it before or after the `group_by()`.

In [ ]:
big_states = (
    orders.group_by(F.col("users.state").alias("state"))
    .agg(F.measure(REVENUE).alias("revenue"))
    .filter(F.measure(REVENUE) > 60_000)
    .sort(F.col("revenue").desc())
)
big_states.show()

In [ ]:
print(big_states.explain())

The `having:` line is the server's, not ours. Six groups came back; 21 were grouped.

---

## 4 · Mixed aggregation — where the DAG appears

Now mix a **governed measure** with an **ad-hoc aggregation** (`F.count_distinct`, which has no
server-side definition) in one `agg()`. The plan cannot be one query, so omniframes splits it:

* the governed half stays **tier 1**;
* the ad-hoc half becomes **tier 2** — SQL whose `FROM` is a governed sub-query
  (`staticQueryReferences`), so the raw rows never leave the warehouse;
* the two halves are aligned **locally** on the group key.

In [ ]:
mixed = orders.group_by(F.col("users.state").alias("state")).agg(
    F.measure(REVENUE).alias("revenue"),
    F.count_distinct("users.id").alias("buyers"),
)
mixed.sort(F.col("buyers").desc()).show(5)

In [ ]:
print(mixed.explain())

Two remote steps, one of each tier, then `align-join`. That align-join deliberately pairs the two
halves' **NULL-state groups with each other** — they are the same group — which is the opposite
of what a user-written `join()` does (§7). Note the tier-2 reference is `(unlimited)`: a silently
paged input to an aggregate is a wrong answer, not a truncated page, and it costs nothing because
the `GROUP BY` happens in the warehouse.

---

## 5 · A UDF — the honest local fallback

`F.udf(fn)` wraps any Python function and applies it row by row. Nothing about a UDF is
expressible on the wire, so **everything from the UDF up runs here** — and `explain()` says so
before you pay for it. The query underneath is still pushed down as far as it goes.

In [ ]:
def region(state):
    """Map a state onto a coarse region — the sort of thing no semantic model has."""
    if state is None:
        return "unknown"
    if state in {"California", "Oregon", "Washington"}:
        return "west"
    if state in {"New York", "New Jersey", "Massachusetts"}:
        return "east"
    return "other"


regions = (
    orders.group_by(F.col("users.state").alias("state"))
    .agg(F.measure(REVENUE).alias("revenue"))
    .with_column("region", F.udf(region)("state"))
    .sort(F.col("revenue").desc())
    .limit(6)
)
regions.show()

In [ ]:
print(regions.explain())

`Local [arrow compute]` names the function (`region(state)`), the sort and the limit that had to
follow it down. That is the trade the plan is asking you to accept: 21 aggregated rows come back
and Python touches each one. Had the UDF sat over a raw scan instead, the remote step would say
`limit: 50000` — same honesty, much bigger bill.

---

## 6 · Raw SQL as an entry point

`session.read.sql(...)` sends **your** statement on the model's connection (needs `QUERY_SQL`).
It always travels with `rewriteSql: false` — the marker without which the server silently ignores
the SQL and answers a question nobody asked.

The SQL is **opaque**: omniframes did not write it and will never re-derive it, so every
operation on top runs in the local engine over its result. The plan says exactly that.

In [ ]:
CATEGORY_SQL = f"""
SELECT p.category         AS category,
       COUNT(*)           AS items,
       SUM(oi.sale_price) AS revenue
FROM {SCHEMA}order_items oi
LEFT JOIN {SCHEMA}products p ON p.id = oi.product_id
GROUP BY 1
""".strip()

categories = (
    session.read.sql(MODEL, CATEGORY_SQL)
    .select("category", "items", "revenue")
    .filter(F.col("category").is_not_null())
    .sort(F.col("revenue").desc())
    .limit(5)
)
categories.show()

In [ ]:
print(categories.explain())

`rewriteSql: false` is on the plan line, and the projection, filter, sort and limit are all under
`Local [arrow compute]` — none of them was pushed into somebody else's SQL.

---

## 7 · Cross-frame join — with SQL NULL semantics

Two frames, two independent queries (they may even be different tiers, as here: a governed
aggregate joined to a raw-SQL job), combined in this process — the query API takes one query at a
time.

The join follows **SQL**: a NULL key matches nothing, not even another NULL. The governed side
has a NULL-state group (buyers with no state, plus the ~1 % of order items whose `user_id`
matches no user at all, because the topic's joins are LEFT joins); the SQL side excludes NULLs.
So an `inner` join drops that group and a `left` join keeps it, unmatched.

In [ ]:
RESIDENTS_SQL = f"""
SELECT u.state  AS "users.state",
       COUNT(*) AS residents
FROM {SCHEMA}users u
WHERE u.state IS NOT NULL
GROUP BY 1
""".strip()

states_sql = session.read.sql(MODEL, RESIDENTS_SQL)
revenue_by_state = orders.group_by("users.state").agg(F.measure(REVENUE).alias("revenue"))

inner = revenue_by_state.join(states_sql, "users.state", "inner")
left = revenue_by_state.join(states_sql, "users.state", "left")

inner.sort(F.col("revenue").desc()).show(5)
print(f"grouped states: {revenue_by_state.count()}   inner: {inner.count()}   left: {left.count()}")

In [ ]:
print(inner.explain())

21 → 20 → 21. The row the inner join dropped is the NULL-state group; the left join keeps it with
`residents` NULL. Nothing was silently reconciled.

---

## 8 · Totals

`with_totals()` asks Omni for its own grand-total row and appends it with a derived `row_type`
column. It **re-aggregates over every row the query touched** — post-filter, pre-limit — so it is
not the sum of the values above it. That is the entire reason to ask the server for it rather
than summing the page yourself.

In [ ]:
totals = (
    orders.group_by(F.col("order_items.status").alias("status"))
    .agg(F.measure(REVENUE).alias("revenue"), F.measure(ORDERS).alias("orders"))
    .sort(F.col("revenue").desc())
    .with_totals()
)
totals.show()

In [ ]:
print(totals.explain())

`totals: column_totals [::total::]` on the wire; `row_type` derived on this side (the run
endpoint has no such column — it flags totals rows with reserved indicator columns that never
reach you).

---

## 9 · Write it out

`df.write` runs the query and writes the **normalized, aliased** result — Parquet keeps the
decimal columns exact, which is why a frame of money should not be a CSV.

In [ ]:
import tempfile

import pyarrow.parquet as pq

out = Path(tempfile.mkdtemp(prefix="omniframes-demo-")) / "revenue_by_state.parquet"
mixed.write.parquet(out)

print(f"wrote {out} ({out.stat().st_size:,} bytes)")
print(pq.read_table(out).schema)

`df.write.csv(path)` is the other writer; both take the small set of options that matter.

---

## 10 · Hand off to pandas

`to_pandas()` is the exit. (`to_arrow()` / `collect()` give you the Arrow table zero-copy, and
`to_polars()` is one `pip install 'omniframes[polars]'` away.)

In [ ]:
frame = mixed.sort(F.col("revenue").desc()).limit(8).to_pandas()

print(frame.dtypes.to_string())
frame

---

## Where next

* **`docs/quickstart.md`** — install, auth, first query.
* **`docs/mental-model.md`** — measures vs. aggregations, limits and truncation, alias semantics,
  `between()`, and SQL NULL semantics, in one honest page.
* **`docs/offline-testing.md`** — the fake, the bench dataset and the six test lanes.
* **`docs/DESIGN.md`**, **`docs/HYBRID.md`**, **`docs/SQLTIER.md`** — the design behind the tiers.

This notebook is executed headlessly against the fake by
`tests/e2e/test_demo_notebook.py` on every CI run, so it cannot rot unnoticed. **Commit it with
its outputs cleared.**